# Company OS en Kaggle — celda única

Este notebook existe como respaldo documentado. El flujo oficial es **una sola celda**
pegada en un notebook nuevo de Kaggle (la celda de abajo es la canónica).

**Secretos requeridos** en Kaggle (Add-ons → Secrets):
- `github` → token con acceso al repo
- `NVIDIA_API_KEY` → key de NVIDIA

La celda sincroniza el repo a la última versión, escribe el `.env`, levanta el
dashboard y lo expone con un túnel de Cloudflare. Al final imprime la URL pública
para abrir desde el iPhone. **No compartir la URL. Apagar el kernel al terminar.**


In [ ]:
# ================================================================
# COMPANY-OS · Celda única para Kaggle
# Pegar en una celda y ejecutar. Muestra la URL pública del
# dashboard para abrir desde el iPhone. Re-ejecutar trae siempre
# la última versión del repo.
# Requiere dos secretos en Kaggle (Add-ons → Secrets):
#   "github"          → token con acceso al repo
#   "NVIDIA_API_KEY" → key de NVIDIA
# ================================================================

# ---------------- CONFIGURACIÓN ----------------
MODELO_BASE = "nvidia/nemotron-3-super-120b-a12b"
MODELO_RAZONAMIENTO = "moonshotai/kimi-k3"
MODELO_LIGERO = "nvidia/nemotron-3.5-lightning-30b-a3b"
PUERTO = 8080

import os, re, sys, json, time, shutil, socket, subprocess, urllib.request

REPO = "enrrutador/company-os"
DIR = "/kaggle/working/company-os"
ENV_FILE = os.path.join(DIR, "infraestructura", "litellm", ".env")

def sh(cmd, cwd=None, timeout=180):
    return subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True,
                          text=True, timeout=timeout)

def puerto_libre(p):
    s = socket.socket()
    try:
        s.bind(("127.0.0.1", p))
        return True
    except OSError:
        return False
    finally:
        s.close()

# ---------------- 1. Secretos ----------------
from kaggle_secrets import UserSecretsClient
_secretos = UserSecretsClient()
sec_github = _secretos.get_secret("github")
sec_nvidia = _secretos.get_secret("NVIDIA_API_KEY")
assert sec_github and sec_nvidia, "Faltan secretos: 'github' y 'NVIDIA_API_KEY'"

# ---------------- 2. Repo siempre actualizado ----------------
print("== Sincronizando repo ==")
if os.path.isdir(os.path.join(DIR, ".git")):
    # limpiar URL con token de clones viejos y traer última versión
    sh("git remote set-url origin https://github.com/enrrutador/company-os", cwd=DIR)
    sh(f"git remote set-url origin https://{sec_github}@github.com/{REPO}", cwd=DIR)
    r = sh("git fetch --depth 1 origin main && git reset --hard origin/main", cwd=DIR)
    sh("git remote set-url origin https://github.com/enrrutador/company-os", cwd=DIR)
    if r.returncode != 0:
        print("fetch falló → re-clonando")
        shutil.rmtree(DIR, ignore_errors=True)
if not os.path.isdir(os.path.join(DIR, ".git")):
    r = sh(f"git clone --depth 1 https://{sec_github}@github.com/{REPO} {DIR}")
    r.check_returncode()
    sh("git remote set-url origin https://github.com/enrrutador/company-os", cwd=DIR)
BUILD_REPO = sh("git rev-parse --short HEAD", cwd=DIR).stdout.strip()
print("build repo:", BUILD_REPO)

# ---------------- 3. Dependencias + .env ----------------
print("== Dependencias ==")
sh(f"{sys.executable} -m pip install -q --break-system-packages "
   f"-r runtime/requirements.txt", cwd=DIR)
print("== .env ==")
os.makedirs(os.path.dirname(ENV_FILE), exist_ok=True)
with open(ENV_FILE, "w") as f:
    f.write("LITELLM_BASE_URL=https://integrate.api.nvidia.com/v1\n")
    f.write(f"LITELLM_MASTER_KEY={sec_nvidia}\n")
    f.write(f"MODELO_EMPRESA_BASE={MODELO_BASE}\n")
    f.write(f"MODELO_EMPRESA_RAZONAMIENTO={MODELO_RAZONAMIENTO}\n")
    f.write(f"MODELO_EMPRESA_LIGERO={MODELO_LIGERO}\n")

# ---------------- 4. Matar dashboard/túnel viejos ----------------
print("== Liberando puerto ==")
sh(f'pkill -f "servidor.py --puerto {PUERTO}"')
sh('pkill -f "cloudflared tunnel"')
for _ in range(30):
    if puerto_libre(PUERTO):
        break
    time.sleep(1)

# ---------------- 5. Dashboard ----------------
print("== Dashboard ==")
subprocess.Popen([sys.executable, "dashboard/servidor.py", "--puerto", str(PUERTO)],
                 cwd=DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    if not puerto_libre(PUERTO):
        break
    time.sleep(1)
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PUERTO}/api/empresa",
                                timeout=10) as r:
        build_srv = json.load(r).get("build")
    print("build servidor:", build_srv,
          "→ OK" if build_srv == BUILD_REPO else "→ ⚠ DESFASAJE")
except Exception as e:
    print("⚠ /api/empresa no responde:", str(e)[:100])

# ---------------- 6. Smoke test (no bloquea el túnel) ----------------
print("== Smoke test ==")
try:
    req = urllib.request.Request(
        f"http://127.0.0.1:{PUERTO}/api/ejecutar",
        data=json.dumps({"slug": "gerente-general",
                         "tarea": "Respondé con una sola palabra: OK"}).encode(),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=180) as r:
        d = json.load(r)
    print("smoke:", "OK" if d.get("ok") else "falló: " + str(d.get("error"))[:120])
except Exception as e:
    print("smoke: salteado (no bloqueante):", type(e).__name__, str(e)[:100])

# ---------------- 7. Túnel público ----------------
print("== Túnel ==")
r = sh("command -v cloudflared")
CLOUDFLARED = "cloudflared" if r.returncode == 0 else "/tmp/cloudflared"
if CLOUDFLARED == "/tmp/cloudflared" and not os.path.exists(CLOUDFLARED):
    print("instalando cloudflared…")
    sh("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/"
       "cloudflared-linux-amd64 -o /tmp/cloudflared && chmod +x /tmp/cloudflared")
subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PUERTO}"],
                 stdout=open("/tmp/tunel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
                  open("/tmp/tunel.log").read())
    if m:
        url = m.group(0)
        break
print()
if url:
    print("══════════════════════════════════════════════════")
    print("  Dashboard:", url)
    print("══════════════════════════════════════════════════")
    print("No compartas la URL. Apagá el kernel al terminar.")
else:
    print("⚠ sin URL de túnel. Log:")
    print(open("/tmp/tunel.log").read()[-1500:])
